# U.S. Data Center Growth and Resource Demand Forecast

This notebook runs the full pipeline from data loading through 2030 forecasts. All the actual work happens in the src modules -- this is just the place to run things and see results.

Charts go to:
- `viz/eda/national/` and `viz/eda/state/` -- what happened 1990 to 2020
- `viz/forecast/national/` and `viz/forecast/state/` -- where things are going through 2030

In [2]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
%matplotlib inline

from src.panel          import build_panel, us_totals
from src.features       import add_features, DC_FEATS, ELEC_FEATS, WATER_FEATS
from src.models         import train_all_models, tune_model, score
from src.forecast       import rollout, us_forecast
from src.eda_plots      import run_all_eda
from src.forecast_plots import run_all_forecast_plots

## 1. Load data

Builds a state by year panel. DC openings are capped at 2020 since that is our confirmed data. Electricity runs to 2024 and we keep it that far because the post-2020 electricity growth is used as a signal during the forecast rollout -- it tells the model what DC demand looked like during the AI buildout years even though we do not have confirmed facility counts yet.

In [ ]:
panel = build_panel(
    '../data/datacenters_clean.csv',
    '../data/electricity_clean.csv',
    '../data/water_clean.csv',
)
us = us_totals(panel)  # national aggregates capped at 2020 for EDA

dc_count = int(panel[panel['year'] <= 2020]['openings'].sum())
print(f"{len(panel):,} state-year rows  |  {panel['state_abbrev'].nunique()} states  |  {panel['year'].min()} to {panel['year'].max()}")
print(f"{dc_count:,} confirmed DC openings through 2020")
panel.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/datacenters_clean.csv'

## 2. EDA -- what the data looks like

All charts saved to viz/eda/. National charts show the full 1990 to 2020 arc. State charts show the six biggest markets with DCs, electricity, and water stacked on a shared timeline so you can see how they move together. Milestone markers show key moments in cloud and internet history.

In [3]:
run_all_eda(panel, us)
print("EDA charts saved.")

EDA charts saved.


In [4]:
# quick inline view of the national picture
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

annual = panel[panel['year'] <= 2020].groupby('year')['openings'].sum()
axes[0].bar(annual.index, annual.values, color='#1D4ED8', alpha=0.7)
axes[0].plot(annual.index, annual.rolling(3, min_periods=1).mean(), color='#EA580C', lw=2, label='3yr avg')
axes[0].set_title('Annual DC Openings')
axes[0].legend()

ue = us.dropna(subset=['electricity_usage_mwh'])
axes[1].fill_between(ue['year'], ue['electricity_usage_mwh']/1e9, color='#DBEAFE', alpha=0.8)
axes[1].plot(ue['year'], ue['electricity_usage_mwh']/1e9, color='#1D4ED8', lw=2)
axes[1].set_title('US Commercial Electricity (B MWh)')

uw = us.dropna(subset=['water_usage_mgal'])
axes[2].fill_between(uw['year'], uw['water_usage_mgal']/1e3, color='#DCFCE7', alpha=0.8)
axes[2].plot(uw['year'], uw['water_usage_mgal']/1e3, color='#15803D', lw=2)
axes[2].set_title('US Water Usage (B gallons)')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.3)

plt.suptitle('National Overview 1990 to 2020', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature engineering

Lag features 1 to 3 years back, rolling averages, exponential smoothing on openings, electricity growth rate both annual and 3-year rolling, state rank by total DC count, and a national momentum feature. The electricity growth features are the main improvement over a naive lag model -- they pick up the acceleration signal during the forecast period.

In [5]:
featured = add_features(panel)

print(f"Feature matrix: {featured.shape}")
print(f"DC features ({len(DC_FEATS)}): {DC_FEATS}")
print(f"Electricity features: {ELEC_FEATS}")
print(f"Water features: {WATER_FEATS}")

Feature matrix: (1785, 21)
DC features (14): ['open_lag1', 'open_lag2', 'open_lag3', 'cum_lag1', 'cum_lag2', 'roll3', 'roll5', 'open_ema3', 'elec_lag1', 'elec_growth', 'elec_growth_3yr', 'year_num', 'state_rank', 'national_momentum']
Electricity features: ['cum_lag1', 'elec_lag1', 'elec_growth', 'year_num', 'state_rank']
Water features: ['cum_lag1', 'water_lag1', 'year_num', 'state_rank']


## 4. Train models

Training on pre-2016, testing on 2016 to 2020. That test window includes the post-2015 surge which is the hardest thing to predict.

DC openings use a two-stage cascade -- a classifier first decides if any DCs will open in a state-year, then a regressor predicts how many. Without this split the regressor gets dominated by all the zero rows and undershoots the active markets. Recent years 2015 to 2020 get 3 to 4x sample weight since the cloud era dynamics are what we actually care about forecasting.

In [6]:
models, test_dfs = train_all_models(featured, DC_FEATS, ELEC_FEATS, WATER_FEATS)

print("DC Openings -- test period 2016 to 2020:")
score(test_dfs['dc']['openings'].values, test_dfs['dc']['pred_openings'].values)

print("\nElectricity:")
score(test_dfs['elec']['electricity_usage_mwh'].values, test_dfs['elec']['pred_elec'].values)

print("\nWater:")
score(test_dfs['water']['water_usage_mgal'].values, test_dfs['water']['pred_water'].values)

DC Openings -- test period 2016 to 2020:
MAE=2.378  RMSE=4.669  R2=0.3432

Electricity:
MAE=535898.714  RMSE=1515070.105  R2=0.9971

Water:
MAE=297.458  RMSE=693.873  R2=0.9954


{'mae': 297.45820585725863,
 'rmse': np.float64(693.8733755540335),
 'r2': 0.9954040794474995}

In [7]:
# national year-by-year is easier to read than per-state accuracy numbers
print('National DC openings -- actual vs predicted by year')
td = test_dfs['dc']
for yr in range(2016, 2021):
    a = int(td[td['year'] == yr]['openings'].sum())
    p = td[td['year'] == yr]['pred_openings'].sum()
    print(f'  {yr}:  actual={a:3d}   predicted={p:5.0f}')

National DC openings -- actual vs predicted by year
  2016:  actual=165   predicted=  179
  2017:  actual=129   predicted=  138
  2018:  actual=199   predicted=  159
  2019:  actual=153   predicted=  177
  2020:  actual=142   predicted=  165


## 5. Hyperparameter tuning (optional)

Uses RandomizedSearchCV with TimeSeriesSplit so CV folds respect temporal order. Takes around 3 minutes.

In [8]:
run_tuning = False

if run_tuning:
    train = featured[(featured['year'] < 2016) & (featured['year'] <= 2020)].dropna(subset=DC_FEATS)
    best_params, best_mae = tune_model(train[DC_FEATS].values, train['openings'].values, n_iter=30)
    print('Best params:', best_params)
    print(f'Best CV MAE: {best_mae:.3f}')
else:
    print('Skipped. Set run_tuning = True to run the search.')

Skipped. Set run_tuning = True to run the search.


## 6. Forecast 2021 to 2030

Autoregressive rollout -- each year feeds the next. For 2021 to 2024 the actual electricity data is injected directly so the model sees the real post-COVID and AI-driven demand surge. After 2024 the electricity model predicts its own next-year values, and those predictions feed back as features for the DC model -- no fixed growth rates anywhere.

In [9]:
fc_state = rollout(models, panel)
fc_us    = us_forecast(fc_state)

print('National forecast:')
print(fc_us[['year', 'openings', 'cum_dcs']].to_string(index=False))

National forecast:
 year  openings  cum_dcs
 2021     173.0   1785.0
 2022     198.0   1983.0
 2023     164.0   2147.0
 2024     150.0   2297.0
 2025     163.0   2460.0
 2026     198.0   2658.0
 2027     181.0   2839.0
 2028     163.0   3002.0
 2029     164.0   3166.0
 2030     176.0   3342.0


In [10]:
new_dcs    = int(fc_state['openings'].sum())
total_2030 = int(panel[panel['year'] <= 2020]['openings'].sum()) + new_dcs
elec_2020  = us[us['year'] == 2020]['electricity_usage_mwh'].values[0]
elec_2030  = fc_us[fc_us['year'] == 2030]['electricity_usage_mwh'].values[0]
water_2020 = us[us['year'] == 2020]['water_usage_mgal'].values[0]
water_2030 = fc_us[fc_us['year'] == 2030]['water_usage_mgal'].values[0]

print(f"New DCs projected 2021 to 2030:  {new_dcs:,}")
print(f"Total U.S. DCs by 2030:          {total_2030:,}")
print(f"Electricity change vs 2020:      {(elec_2030/elec_2020 - 1)*100:+.1f}%")
print(f"Water usage change vs 2020:      {(water_2030/water_2020 - 1)*100:+.1f}%")
print()

top3 = fc_state[fc_state['year'] == 2030].sort_values('cum_dcs', ascending=False).head(3)
print('Top 3 states by cumulative DCs in 2030:')
for _, r in top3.iterrows():
    print(f"  {r['state_abbrev']}: {int(r['cum_dcs']):,} cumulative facilities")

New DCs projected 2021 to 2030:  1,730
Total U.S. DCs by 2030:          3,342
Electricity change vs 2020:      +36.3%
Water usage change vs 2020:      +3.9%

Top 3 states by cumulative DCs in 2030:
  TX: 339 cumulative facilities
  CA: 248 cumulative facilities
  VA: 247 cumulative facilities


## 7. Generate forecast charts

Model evaluation, feature importance, national and state forecasts, and the holistic combined views. All saved to viz/forecast/. The holistic charts show DCs, electricity, and water together on a shared timeline with the forecast extending to 2030 and milestone markers for the AI era.

In [11]:
run_all_forecast_plots(
    hist_panel=panel,
    hist_us=us,
    fc_state=fc_state,
    fc_us=fc_us,
    test_dfs=test_dfs,
    models=models,
    DC_FEATS=DC_FEATS,
    ELEC_FEATS=ELEC_FEATS,
)
print("Forecast charts saved.")

Forecast charts saved.


In [12]:
# list everything that was saved
from pathlib import Path
for p in sorted(Path('viz').rglob('*.png')):
    print(p)

viz\eda\national\correlations.png
viz\eda\national\dc_growth.png
viz\eda\national\electricity.png
viz\eda\national\holistic_overview.png
viz\eda\national\water.png
viz\eda\state\growth_curves.png
viz\eda\state\holistic_ca.png
viz\eda\state\holistic_fl.png
viz\eda\state\holistic_ga.png
viz\eda\state\holistic_oh.png
viz\eda\state\holistic_tx.png
viz\eda\state\holistic_va.png
viz\eda\state\rankings.png
viz\eda\state\resource_scatter.png
viz\forecast\national\dc_forecast.png
viz\forecast\national\holistic_forecast.png
viz\forecast\national\model_fit_dc.png
viz\forecast\national\model_fit_elec.png
viz\forecast\national\model_fit_water.png
viz\forecast\national\resource_forecast.png
viz\forecast\national\summary_table_2030.png
viz\forecast\state\dc_forecast_grid.png
viz\forecast\state\holistic_forecast_ca.png
viz\forecast\state\holistic_forecast_fl.png
viz\forecast\state\holistic_forecast_ga.png
viz\forecast\state\holistic_forecast_oh.png
viz\forecast\state\holistic_forecast_tx.png
viz\forec

## 8. Random Forest vs cascade (DC openings)

Trains a **RandomForestRegressor** in `src/treemodel.py` using **DC features plus explicit electricity and water usage** (`electricity_usage_mwh`, `water_usage_mgal`, `water_lag1`, alongside the existing elec signals in `DC_FEATS`). The cascade in `src/models.py` is unchanged (still `DCOpeningsModel` on `DC_FEATS` only).

**Running this section:** If you already ran sections 1–7 in this kernel, variables are reused. If `featured` / `models` / `panel` are missing (fresh kernel or only this section), the first code cell reloads data and retrains the cascade automatically.

Metrics and plots use **aligned** test rows where every RF input is non-null (stricter than the cascade-only split). Figures are saved under `viz/model_compare/`.

In [ ]:
from src.treemodel import (
    reload_pipeline_for_section8,
    openings_rf_feature_columns,
    fit_openings_random_forest,
    build_aligned_test_predictions,
    build_history_prediction_frame,
    score_dc_comparison,
)

try:
    featured
    models
    test_dfs
    panel
except NameError:
    panel, featured, models, test_dfs = reload_pipeline_for_section8()

RF_OPENINGS_FEATS = openings_rf_feature_columns()
print(len(RF_OPENINGS_FEATS), "RF features (incl. electricity & water levels):", RF_OPENINGS_FEATS)

rf_openings_model, _, _ = fit_openings_random_forest(featured)

test_df, y_true, pred_rf, pred_cascade = build_aligned_test_predictions(
    featured, models, rf_openings_model
)

comparison_df = score_dc_comparison(y_true, pred_rf, pred_cascade)
comparison_df

In [ ]:
from pathlib import Path

from src.model_compare_plots import run_openings_model_comparison_plots

hist_pred = build_history_prediction_frame(featured, models, rf_openings_model)

run_openings_model_comparison_plots(
    y_true,
    pred_cascade,
    pred_rf,
    rf_openings_model,
    RF_OPENINGS_FEATS,
    hist_pred,
    train_cutoff=2016,
)

print("Comparison figures saved under viz/model_compare/")
for p in sorted(Path("viz/model_compare").glob("*.png")):
    print(p)